Paso 1: Cargar y explorar el dataset

In [20]:
import pandas as pd

# Cargar dataset
df = pd.read_csv("trainingData.csv")

# Mostrar primeras filas
print(df.head())

# Dimensiones
print("Shape:", df.shape)

# Clases en FLOOR
print(df['FLOOR'].value_counts())

   WAP001  WAP002  WAP003  WAP004  WAP005  WAP006  WAP007  WAP008  WAP009  \
0     100     100     100     100     100     100     100     100     100   
1     100     100     100     100     100     100     100     100     100   
2     100     100     100     100     100     100     100     -97     100   
3     100     100     100     100     100     100     100     100     100   
4     100     100     100     100     100     100     100     100     100   

   WAP010  ...  WAP520  LONGITUDE      LATITUDE  FLOOR  BUILDINGID  SPACEID  \
0     100  ...     100 -7541.2643  4.864921e+06      2           1      106   
1     100  ...     100 -7536.6212  4.864934e+06      2           1      106   
2     100  ...     100 -7519.1524  4.864950e+06      2           1      103   
3     100  ...     100 -7524.5704  4.864934e+06      2           1      102   
4     100  ...     100 -7632.1436  4.864982e+06      0           0      122   

   RELATIVEPOSITION  USERID  PHONEID   TIMESTAMP  
0          

Paso 2: Preparar los datos

In [21]:
# Eliminar columnas no relevantes
cols_to_drop = ['LONGITUDE','LATITUDE','SPACEID','RELATIVEPOSITION',
                'USERID','PHONEID','TIMESTAMP']

df = df.drop(columns=cols_to_drop)

# Separar X e y
X = df.drop(columns=['FLOOR'])
y = df['FLOOR']

print(X.shape, y.shape)

(19937, 521) (19937,)



Paso 3: Preprocesamiento de las señales WiFi

In [22]:
# Reemplazar 100 por -100
X[X == 100] = -100

Paso 4: Preparación del dataset

In [23]:
def preparar_datos(ruta_csv, target='FLOOR', random_state=42):
    df = pd.read_csv(ruta_csv)

    cols_to_drop = ['LONGITUDE','LATITUDE','SPACEID','RELATIVEPOSITION',
                    'USERID','PHONEID','TIMESTAMP']
    df = df.drop(columns=cols_to_drop)

    X = df.loc[:, 'WAP001':'WAP520']
    y = df[target]

    X[X == 100] = -100

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=random_state
    )

    return X_train, X_val, y_train.values, y_val.values


X_train, X_val, y_train, y_val = preparar_datos("trainingData.csv")
X_test, y_test = X_val, y_val

/tmp/ipykernel_722/4240174982.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[X == 100] = -100
/tmp/ipykernel_722/4240174982.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[X == 100] = -100


Paso 5: Entrenamiento de redes neuronales artificiales (ANN)

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time

# Convertir a tensores
def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Arquitecturas
class ANN1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(520,128),
            nn.ReLU(),
            nn.Linear(128,5) # Changed from 4 to 5 to match 5 unique FLOOR values (0,1,2,3,4)
        )
    def forward(self,x): return self.net(x)

class ANN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(520,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,5) # Changed from 4 to 5 to match 5 unique FLOOR values (0,1,2,3,4)
        )
    def forward(self,x): return self.net(x)

class ANN3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(520,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,5) # Changed from 4 to 5 to match 5 unique FLOOR values (0,1,2,3,4)
        )
    def forward(self,x): return self.net(x)

In [25]:
def entrenar(model, X_train, y_train, X_val, y_val, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    X_train, y_train = to_tensor(X_train, y_train)
    X_val, y_val = to_tensor(X_val, y_val)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        output = model(X_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_output = model(X_val)
            val_loss = criterion(val_output, y_val)

        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss.item())

    return history

In [26]:
def evaluar(model, X_test, y_test):
    X_test = torch.tensor(X_test, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        preds = torch.argmax(outputs, axis=1).numpy()

    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')

    return acc, precision, recall, f1

Paso 6: Tabla resumen de resultados por arquitectura

In [30]:
import pandas as pd

resultados = []

modelos = [ANN1(), ANN2(), ANN3()]

# --- Diagnóstico: Imprimir la salida esperada del modelo y el valor máximo de y_train ---
print(f"Output features of ANN1's last layer: {modelos[0].net[-1].out_features}")
print(f"Max value in y_train: {y_train.max()}")
print(f"Min value in y_train: {y_train.min()}")
# -------------------------------------------------------------------------------------

for i, model in enumerate(modelos):
    start = time.time()

    entrenar(model, X_train, y_train, X_val, y_val)

    end = time.time()

    acc, prec, rec, f1 = evaluar(model, X_test, y_test)

    resultados.append([f"Arquitectura {i+1}", acc, prec, rec, f1, end-start])

tabla = pd.DataFrame(resultados, columns=[
    "Arquitectura","Accuracy","Precision","Recall","F1-score","Tiempo"
])

print(tabla)

Output features of ANN1's last layer: 5
Max value in y_train: 4
Min value in y_train: 0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


     Arquitectura  Accuracy  Precision    Recall  F1-score    Tiempo
0  Arquitectura 1  0.658977   0.703470  0.658977  0.631847  3.150511
1  Arquitectura 2  0.621866   0.667579  0.621866  0.577181  6.125461
2  Arquitectura 3  0.577733   0.583446  0.577733  0.516563  6.418095


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Paso 7: Evaluar el impacto del número de épocas en el mejor modelo

In [31]:
epocas_list = [10,20,30,40,50]
resultados_epocas = []

for ep in epocas_list:
    model = ANN2()  # ejemplo mejor modelo

    start = time.time()
    entrenar(model, X_train, y_train, X_val, y_val, epochs=ep)
    end = time.time()

    acc, prec, rec, f1 = evaluar(model, X_test, y_test)

    resultados_epocas.append([ep, acc, prec, rec, f1, end-start])

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Paso 8: Tabla resumen de resultados por número de épocas

In [32]:
tabla_epocas = pd.DataFrame(resultados_epocas, columns=[
    "Epocas","Accuracy","Precision","Recall","F1-score","Tiempo"
])

print(tabla_epocas)

   Epocas  Accuracy  Precision    Recall  F1-score     Tiempo
0      10  0.341775   0.494299  0.341775  0.238884   4.805619
1      20  0.799398   0.783643  0.799398  0.775377   5.607342
2      30  0.877884   0.837762  0.877884  0.854452   9.593672
3      40  0.908726   0.865152  0.908726  0.884623  12.320150
4      50  0.947844   0.951773  0.947844  0.941798  15.388824


**Preguntas de Análisis**

**1. ¿Cuál considera que fue la mejor arquitectura evaluada? ¿Por qué?**

La mejor arquitectura fue la Arquitectura 1, ya que obtuvo el mayor rendimiento en el conjunto de test con un accuracy de 0.659 y un F1-score de 0.632. Además, fue la más eficiente en términos de tiempo de entrenamiento (3.15 s), lo que indica un buen equilibrio entre desempeño y costo computacional.

**2. ¿Cuál fue la arquitectura con peor desempeño? ¿A qué cree que se debió su bajo rendimiento?**

La arquitectura con peor desempeño fue la Arquitectura 3, con un accuracy de 0.578 y un F1-score de 0.517. Esto puede deberse a que su mayor profundidad no logró generalizar correctamente, posiblemente generando sobreajuste o dificultades en el entrenamiento debido a la complejidad adicional.

**3. ¿Cómo influye el número de capas ocultas en el comportamiento de la red?**

El número de capas ocultas influye directamente en la capacidad del modelo para aprender patrones complejos. En este caso, se observó que aumentar la profundidad no necesariamente mejoró el rendimiento, ya que las arquitecturas más profundas (como la 2 y 3) obtuvieron peores métricas que la más simple. Esto sugiere que un modelo más complejo no siempre generaliza mejor.

**4. ¿Cuál fue la mejor cantidad de épocas para entrenar el mejor modelo? Justifique su elección.**

La mejor cantidad de épocas fue 50, ya que alcanzó el mejor rendimiento con un accuracy de 0.948 y un F1-score de 0.942. Aunque el tiempo de entrenamiento aumentó a 15.39 s, las métricas mejoraron de forma significativa en comparación con menos épocas, lo que justifica el incremento.

**5. ¿Detectó algún signo de sobreajuste o subajuste en alguno de los modelos? ¿Cómo lo identificó?**

Sí, se detectó subajuste en las primeras épocas, especialmente con 10 épocas, donde el modelo obtuvo un F1-score de 0.239, indicando que no había aprendido lo suficiente. A medida que aumentaron las épocas, el rendimiento mejoró considerablemente, lo que sugiere que el modelo necesitaba más entrenamiento para capturar los patrones del dataset.

**6. ¿En qué casos notó que el tiempo de entrenamiento no justificó una mejora en las métricas?**

Entre 40 y 50 épocas, el tiempo de entrenamiento aumentó de 12.32 s a 15.39 s, mientras que la mejora en F1-score fue relativamente menor (de 0.885 a 0.942) en comparación con saltos anteriores. Esto indica que, aunque hubo mejora, el incremento en tiempo comienza a mostrar rendimientos decrecientes.

**7. ¿La arquitectura más profunda fue también la más precisa? ¿Qué conclusiones saca de esto?**

No, la arquitectura más profunda (Arquitectura 3) no fue la más precisa. De hecho, obtuvo el peor desempeño. Esto demuestra que aumentar la complejidad del modelo no garantiza mejores resultados y que es fundamental encontrar un equilibrio adecuado entre profundidad y capacidad de generalización.

**8. ¿Qué métrica considera más importante en este contexto (accuracy, precision, recall, F1-score) y por qué?**

El F1-score es la métrica más importante en este contexto, ya que combina precision y recall en una sola medida equilibrada. En problemas multiclase como este, es fundamental considerar tanto los falsos positivos como los falsos negativos, lo que hace que el F1-score sea más representativo del desempeño real del modelo que el accuracy por sí solo.